In [1]:
pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
from tqdm import tqdm
import time
import pandas as pd

In [3]:
load_dotenv()
api_key = os.getenv("GROQ_API")

In [4]:
client = Groq(api_key=api_key)

In [5]:
def load_similarity_map(file_path, category):
    """
    Converts wide-format similarity CSVs into a lookup dictionary.
    Handles 'Drift/Variant' or 'Drift_Level' as row index (L1-L5).
    """
    if not os.path.exists(file_path):
        return {}
    
    df = pd.read_csv(file_path)
    # Identify the drift level column (handles 'Drift/Variant' or 'Drift_Level')
    drift_col = df.columns[0]
    
    sim_map = {}
    for _, row in df.iterrows():
        drift_level = int(row[drift_col])
        for col in df.columns[1:]:
            # col looks like 'reasoning_1_V1'
            # We want to map it to 'reasoning_1_L1_V1'
            base_part = "_".join(col.split("_")[:-1]) # 'reasoning_1'
            variant_part = col.split("_")[-1]         # 'V1'
            prompt_id = f"{base_part}_L{drift_level}_{variant_part}"
            sim_map[prompt_id] = row[col]
            
    return sim_map

In [6]:
# def build_judge_prompt(category, prompt, response):
#     return f"""
# [INST] You are a highly critical AI Quality Auditor. Your task is to evaluate a Model Response based on a specific User Prompt.

# ### EVALUATION CONTEXT
# - Task Category: {category}
# - Clean Prompt: {prompt}
# - Model Response to Evaluate: {response}

# NOTE: The model may have received a degraded version of this prompt (with typos, missing punctuation, slang, etc.), but you should evaluate the response against the CLEAN task requirements above.

# ### SCORING RUBRIC (0-100)
# - 90-100: Perfect, no errors, highly professional.
# - 70-89: Correct but missing minor nuance or slightly robotic.
# - 40-69: Major factual or logical gaps.
# - 0-39: Completely wrong, hallucinated, or irrelevant.

# ### CRITICAL INSTRUCTIONS
# 1. The 'explanation' must be a single sentence.
# 2. Output MUST be valid JSON. No prose. No intro. No outro.

# ### REQUIRED JSON FORMAT
# {{
#   "factual_correctness": <int>,
#   "logical_correctness": <int>,
#   "reasoning_coherence": <int>,
#   "sentiment_label_correctness": <int>,
#   "manner_of_language": <int>,
#   "explanation": "<string>"
# }}
# [/INST]"""


def build_judge_prompt(category, clean_prompt, drifted_prompt, response, prompt_sim, response_sim):
    return f"""
[INST] You are an Adversarial AI Quality Auditor. Your mission is to determine if a model's "Reasoning Spine" has collapsed due to input noise.

### STABILITY METRICS (OBJECTIVE DATA)
1. **Prompt Similarity (Clean vs. Noisy)**: {prompt_sim:.2f} 
   *(Lower means the model had a much harder task interpreting the intent.)*
2. **Response Divergence (Clean Output vs. Current Output)**: {response_sim:.2f} 
   *(Lower means the model's behavior shifted significantly compared to its own clean-input baseline.)*

### EVALUATION CONTEXT
- Task Category: {category}
- Clean Requirement: {clean_prompt}
- Actual Input Received: {drifted_prompt}
- Model Response: {response}

### SCORING INSTRUCTIONS (THE STABILITY PENALTY)
- **Grounding Rule**: If Response Divergence is below 0.80, the model is exhibiting "Behavioral Instability." You must penalize 'Logical Correctness' and 'Reasoning Coherence' significantly (usually <60).
- **Nuance Check**: Compare the response to the Clean Requirement. If the model ignored a detail because of the noise in the 'Actual Input', score Factual Correctness <50.

### REQUIRED JSON FORMAT
{{
  "factual_correctness": <int>,
  "logical_correctness": <int>,
  "reasoning_coherence": <int>,
  "sentiment_label_correctness": <int>,
  "manner_of_language": <int>
}}
[/INST]"""

In [7]:
ROOT_DIR = r"..\3_prompt_responses\outputs"
SAVE_DIR = r"judge_scores"

os.makedirs(SAVE_DIR, exist_ok=True)

In [8]:
def judge_response_groq(category, clean_prompt, drifted_prompt, response, p_sim, r_sim):
    # Pass metrics to the prompt builder
    judge_prompt = build_judge_prompt(category, clean_prompt, drifted_prompt, response, p_sim, r_sim)
    
    try:
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": "You are a strict grading bot that outputs ONLY valid JSON. Prioritize penalizing instability."},
                {"role": "user", "content": judge_prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"Error calling Groq: {e}")
        return "error"

In [9]:
# def evaluate_specific_category(provider, model_name, bit_level, category):
#     """
#     Processes and evaluates a specific dataset slice.
#     Uses CLEAN PROMPT for fair comparison across drift levels.
#     """
#     # 1. Construct the specific path
#     category_path = os.path.join(ROOT_DIR, provider, f"{model_name}_{bit_level}", category)
#     responses_file = os.path.join(category_path, "responses.json")
    
#     if not os.path.exists(responses_file):
#         print(f"File not found: {responses_file}")
#         return

#     # 2. Load the data
#     with open(responses_file, 'r') as f:
#         responses = json.load(f)
#         # responses = responses[0:10]  # Testing on 10 samples
    
#     # 3. Load ideal prompts for reference
#     ideal_prompts_file = os.path.join(r"..\1_dataset_prep\ideal_prompts.json")  # UPDATE PATH
#     with open(ideal_prompts_file, 'r') as f:
#         ideal_data = json.load(f)
    
#     # Create mapping: base_id -> clean_prompt
#     clean_prompt_map = {}
#     for item in ideal_data['prompts']:
#         clean_prompt_map[item['id']] = item['prompt']

#     results = []
    
#     # 4. Process with judge
#     for item in tqdm(responses, desc=f"Eval: {model_name}-{bit_level}-{category}"):
#         drifted_prompt = item["prompt"]  # What was actually sent to model
#         response = item["response"]
#         base_id = item["base_id"]
        
#         # GET CLEAN PROMPT (the key change!)
#         clean_prompt = clean_prompt_map.get(base_id, drifted_prompt)  # Fallback to drifted if not found
        
#         # Simple retry logic for JSON robustness
#         score = "error"
#         for attempt in range(3):  # Try twice
#             try:
#                 # Pass clean_prompt, not drifted_prompt
#                 raw_score = judge_response_groq(category, clean_prompt, response)
#                 if raw_score and raw_score != "error":
#                     score = json.loads(raw_score)
#                     break 
#             except Exception as e:
#                 continue

#         results.append({
#             "id": item["id"],
#             "base_id": item["base_id"],
#             "category": item["category"],
#             "drift_level": item["drift_level"],
#             "variant": item["variant"],
#             "model": item["model"],
#             "quantization": item["quantization"],
#             "drifted_prompt": drifted_prompt,  # Keep for reference
#             "clean_prompt": clean_prompt,      # What judge saw
#             "scores": score
#         })

#         time.sleep(1)

#     # 5. Save the specific results
#     save_folder = os.path.join(SAVE_DIR, provider, f"{model_name}_{bit_level}", category)
#     os.makedirs(save_folder, exist_ok=True)
#     save_path = os.path.join(save_folder, "judge_scores.json")
    
#     with open(save_path, "w") as f:
#         json.dump(results, f, indent=4)
    
#     print(f"✅ Successfully saved to: {save_path}")

def evaluate_specific_category(provider, model_name, bit_level, category):
    """
    Updated evaluation function incorporating prompt and response similarity.
    """
    # 1. Setup Paths
    model_folder = f"{model_name}_{bit_level}"
    category_path = os.path.join(ROOT_DIR, provider, model_folder, category)
    responses_file = os.path.join(category_path, "responses.json")
    
    # Similarity File Paths
    # Note: Mapping 'qa' category to 'qna' filename as per your structure
    filename_cat = "qna" if category.lower() == "qa" else category.lower()
    
    prompt_sim_file = os.path.join(r"..\2_prompts_cossim\prompt_similarity_outputs", f"{filename_cat}_prompt_similarity.csv")
    resp_sim_file = os.path.join(r"..\3_responses_cossim\response_cossim_results", provider, model_folder, f"{filename_cat}_response_cossim.csv")

    if not os.path.exists(responses_file):
        print(f"File not found: {responses_file}")
        return

    # 2. Load Similarity Maps
    p_sim_map = load_similarity_map(prompt_sim_file, category)
    r_sim_map = load_similarity_map(resp_sim_file, category)

    # 3. Load Ideal Prompts
    ideal_prompts_file = os.path.join(r"..\1_dataset_prep\ideal_prompts.json")
    with open(ideal_prompts_file, 'r') as f:
        ideal_data = json.load(f)
    clean_prompt_map = {item['id']: item['prompt'] for item in ideal_data['prompts']}

    # 4. Process Responses
    with open(responses_file, 'r') as f:
        responses = json.load(f)
        # responses = responses[74:]

    results = []
    for item in tqdm(responses, desc=f"Eval: {model_name}-{bit_level}-{category}"):
        item_id = item["id"]
        drifted_prompt = item["prompt"]
        response = item["response"]
        clean_prompt = clean_prompt_map.get(item["base_id"], drifted_prompt)
        
        # Get Similarity Scores (Default to 100 for L0 or missing data)
        p_sim = p_sim_map.get(item_id, 100.0)
        r_sim = r_sim_map.get(item_id, 100.0)

        # 5. Get Scores from Judge
        score = "error"
        for _ in range(5): # Retry logic
            time.sleep(1)
            try:
                raw_score = judge_response_groq(category, clean_prompt, drifted_prompt, response, p_sim, r_sim)
                if raw_score and raw_score != "error":
                    score = json.loads(raw_score)
                    break 
            except:
                continue

        results.append({
            **item, # Preserve original metadata
            "clean_prompt": clean_prompt,
            "prompt_similarity": p_sim,
            "response_similarity": r_sim,
            "scores": score
        })
        time.sleep(5)

    # 6. Save Results (Modified for Appending)
    save_folder = os.path.join(SAVE_DIR, provider, model_folder, category)
    os.makedirs(save_folder, exist_ok=True)
    save_path = os.path.join(save_folder, "judge_scores.json")
    
    # Check if a results file already exists
    existing_results = []
    if os.path.exists(save_path):
        with open(save_path, "r") as f:
            try:
                existing_results = json.load(f)
            except json.JSONDecodeError:
                existing_results = []

    # Merge new results with existing ones
    # (Optional: Use a set of IDs to prevent duplicates)
    combined_results = existing_results + results
    
    with open(save_path, "w") as f:
        json.dump(combined_results, f, indent=4)
    
    print(f"✅ Evaluation complete. Saved to: {save_path}")

In [10]:
# --- EXAMPLE USAGE ---
evaluate_specific_category(
    provider="microsoft", 
    model_name="Phi-3-mini-4k-instruct", 
    bit_level="4bit", 
    category="reasoning"
)

Eval: Phi-3-mini-4k-instruct-4bit-reasoning:  84%|████████▍ | 134/160 [18:33<03:42,  8.56s/it]

Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199843, Requested 1153. Please try again in 7m10.272s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199839, Requested 1115. Please try again in 6m52.128s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2

Eval: Phi-3-mini-4k-instruct-4bit-reasoning:  84%|████████▍ | 135/160 [18:44<03:55,  9.41s/it]

Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199817, Requested 1138. Please try again in 6m52.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199814, Requested 1160. Please try again in 7m0.768s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dz

Eval: Phi-3-mini-4k-instruct-4bit-reasoning:  85%|████████▌ | 136/160 [18:55<03:53,  9.71s/it]

...

Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199249, Requested 1185. Please try again in 3m7.488s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199247, Requested 1147. Please try again in 2m50.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2d

Eval: Phi-3-mini-4k-instruct-4bit-reasoning:  99%|█████████▉| 159/160 [23:00<00:10, 10.92s/it]

Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199224, Requested 1140. Please try again in 2m37.248s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2dzce2aqhcnf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199222, Requested 1140. Please try again in 2m36.384s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Error calling Groq: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km3h991zf5xa2

Eval: Phi-3-mini-4k-instruct-4bit-reasoning: 100%|██████████| 160/160 [23:11<00:00,  8.70s/it]

✅ Evaluation complete. Saved to: judge_scores\microsoft\Phi-3-mini-4k-instruct_4bit\reasoning\judge_scores.json
